In [ ]:
# ========= install (Kaggle) =========
import sys, subprocess, os, math, json, re, glob, io, base64, textwrap, traceback
subprocess.check_call([sys.executable, "-m", "pip", "install", "-qU", "google-genai", "open_clip_torch", "faiss-cpu", "pandas", "numpy", "matplotlib", "tqdm"])

# ========= imports =========
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path
from tqdm.auto import tqdm
import faiss, sqlite3, contextlib, unicodedata, shutil, subprocess, html

# Gemini
from google import genai
from google.genai.types import GenerateContentConfig, Schema, Type

# Display helpers
from IPython.display import display, HTML
plt.rcParams["figure.dpi"] = 120

In [ ]:
# ======================= FLEXIBLE MULTI-ASPECT SEARCH (CLIP + ULTRA-WINDOWED TRANSCRIPT) =======================
# - Tự động (hoặc thủ công) sinh các "query aspects", "names" và "seed keywords" bằng Gemini qua thư viện google-genai
# - Hỗ trợ Batch-1 (Lxx), Batch-2 (Kxx) HOẶC cả hai
# - CLIP window search (ví dụ 40s), rồi rerank transcript cực nhanh trong [t0-Δ, t1+Δ] per-video
# - UI hiển thị grid có dãn cách nhãn, thời gian "win" hiển thị dạng mm:ss
# ==============================================================================================================

import os, re, glob, gc, sqlite3, time, json, textwrap, concurrent.futures
from pathlib import Path
from collections import deque
from typing import List, Dict, Tuple, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import HTML, display

# ------------------------------------ DATASET PATHS ------------------------------------
# Batch-1
B1 = {
    "clip_dir": "/kaggle/input/hcm2025-metadata/clip-features-32-aic25-b1/clip-features-32",
    "map_csv":  "/kaggle/input/hcm2025-metadata/map-keyframes-aic25-b1/map-keyframes",
    "kf_root":  "/kaggle/input/hcmc2025-batch1-keyframes",    # /Lxx/Vyyy/NNN.jpg
    "vid_root": "/kaggle/input/hcm25-batch1-videos/video",    # /{video_id}.mp4
    "keep_dir": "/kaggle/input/hcmc25-preprocessing/clip_keep_idxs"  # optional
}

# Batch-2  (theo yêu cầu)
B2 = {
    "clip_dir": "/kaggle/input/hcmc-ai25-clip-batch2/features_b2/clip-features-32",
    "map_csv":  "/kaggle/input/hcmc-ai25-batch2-keyframe/map-keyframes",
    "kf_root":  "/kaggle/input/hcmc-ai25-batch2-keyframe/keyframes",   # /Kxx/Vyyy/NNN.jpg
    "vid_root": "/kaggle/input/hcmai25-batch2-videos/video",           # /{video_id}.mp4
    "keep_dir": "/kaggle/input/hcmc-ai25-batch2-preprocessing/clip_keep_idxs"  # đã lọc dedup
}

# Text DBs (FTS5) cho transcript
B1_DB_PATH = "/kaggle/input/hcmc-ai25-batch2-bm25/batch1_text_bm25.db"
B2_DB_PATH = "/kaggle/input/hcmc-ai25-batch2-bm25/batch2_text_bm25.db"

# ------------------------------------ GLOBAL CONFIGS -----------------------------------
# Chọn dataset: {"b1"}, {"b2"}, hoặc {"b1","b2"}
DATASETS = {"b1", "b2"}

# Kaggle key
from kaggle_secrets import UserSecretsClient
# Đổi nhãn theo bạn: vd "GEMINI_API_KEY" hoặc "your-secret-label"
_SECRET_LABEL = "GOOGLE_API_KEY"      # ← hoặc "your-secret-label"
_kaggle_key   = UserSecretsClient().get_secret(_SECRET_LABEL)

# Bật dùng LLM (Gemini) để sinh aspects/seed; tắt = dùng định nghĩa thủ công phía dưới
GEMINI_MODEL     = "gemini-2.5-flash"
GEMINI_MAX_TOKENS = 10_000              # theo yêu cầu
GEMINI_API_KEY   = os.environ.get("GOOGLE_API_KEY", _kaggle_key or "")  # set trong Kaggle Secrets nếu cần
USE_LLM_ASPECTS = bool(GEMINI_API_KEY)
print(f"[Gemini] key loaded: {bool(GEMINI_API_KEY)} | USE_LLM_ASPECTS={USE_LLM_ASPECTS}")

WINDOW_SEC   = 40.0
SKIP_SEC     = 0.0
DELTA_SEC    = 20.0
TOP_VIDEOS   = 10
LIMIT_VIDEOS = None
EXCLUDE_SER  = set()   # ví dụ: {"L25"} nếu cần né intro

# ------------------------------------ UTILS --------------------------------------------
def l2_normalize_rows(x: np.ndarray) -> np.ndarray:
    x = x.astype("float32", copy=False)
    n = np.linalg.norm(x, axis=1, keepdims=True) + 1e-12
    return x / n

def sec_to_mmss(t: float) -> str:
    if t < 0 or not np.isfinite(t): return "00:00"
    m = int(t // 60)
    s = int(round(t - m*60))
    if s == 60: m, s = m + 1, 0
    return f"{m:02d}:{s:02d}"

def dataset_for_video(video_id: str) -> str:
    # Lxx_* thuộc b1; Kxx_* thuộc b2
    return "b1" if video_id.startswith("L") else "b2"

def keyframe_path(video_id: str, frame_id: int) -> str:
    """
    Trả về đường dẫn keyframe cho video_id (Lxx_* hoặc Kxx_*).
    - B1 (Lxx): thường 3 chữ số -> 001.jpg
    - B2 (Kxx): thường 4 chữ số -> 0001.jpg
    Hàm này thử lần lượt các khả năng (3/4 chữ số, jpg/jpeg/png; frame_id và frame_id-1).
    """
    D = B1 if dataset_for_video(video_id) == "b1" else B2
    L, V = video_id.split("_")
    base = f"{D['kf_root']}/{L}/{V}"

    # Các biến thể thử lần lượt
    digits = (3, 4) if dataset_for_video(video_id) == "b1" else (4, 3)
    frames_try = (frame_id, max(0, frame_id - 1))   # đề phòng thư mục đánh số từ 0
    exts = (".jpg", ".jpeg", ".png")

    for d in digits:
        for fr in frames_try:
            for ext in exts:
                p = f"{base}/{fr:0{d}d}{ext}"
                if os.path.exists(p):
                    return p

    # Không tìm thấy -> trả về 1 path mặc định để safe_imread() vẽ placeholder
    return f"{base}/{frame_id:0{digits[0]}d}.jpg"

def npy_path_for_video(video_id: str) -> str:
    D = B1 if dataset_for_video(video_id) == "b1" else B2
    return f"{D['clip_dir']}/{video_id}.npy"

def map_csv_path_for_video(video_id: str) -> str:
    D = B1 if dataset_for_video(video_id) == "b1" else B2
    return f"{D['map_csv']}/{video_id}.csv"

def keep_idxs_path_for_video(video_id: str) -> str:
    D = B1 if dataset_for_video(video_id) == "b1" else B2
    return f"{D['keep_dir']}/{video_id}_keep_idxs.npy"

def safe_imread(path: str) -> Image.Image:
    try: return Image.open(path).convert("RGB")
    except: return Image.fromarray(np.full((360, 640, 3), 230, dtype=np.uint8))

def list_videos_from_dirs(clip_dirs: List[str],
                          limit: Optional[int] = None,
                          include_series: Optional[set] = None,
                          exclude_series: Optional[set] = None,
                          include_regex: Optional[str] = None,
                          exclude_regex: Optional[str] = None,
                          include_ids: Optional[set] = None,
                          exclude_ids: Optional[set] = None) -> List[str]:
    files = []
    for d in clip_dirs:
        if d and os.path.isdir(d):
            files.extend(glob.glob(os.path.join(d, "*.npy")))
    vids = [os.path.splitext(os.path.basename(p))[0] for p in sorted(set(files))]
    def keep(vid: str) -> bool:
        sid = vid.split("_", 1)[0]
        if include_series and sid not in include_series: return False
        if exclude_series and sid in exclude_series:   return False
        if include_ids and vid not in include_ids:     return False
        if exclude_ids and vid in exclude_ids:         return False
        if include_regex and not re.search(include_regex, vid): return False
        if exclude_regex and re.search(exclude_regex, vid):     return False
        # L/K filter theo DATASETS
        ds = dataset_for_video(vid)
        if ds not in DATASETS: return False
        return True
    vids = [v for v in vids if keep(v)]
    return vids[:limit] if limit else vids

def load_times_for_video(video_id: str, n_feats: int) -> np.ndarray:
    csv_path = map_csv_path_for_video(video_id)
    if not os.path.exists(csv_path):
        # fallback: tuyến tính
        return np.linspace(0.0, float(max(n_feats-1, 1)), num=n_feats, dtype=float)

    df = pd.read_csv(csv_path)
    t_all = df["pts_time"].to_numpy().astype(float)

    keep_path = keep_idxs_path_for_video(video_id)
    if os.path.exists(keep_path):
        try:
            keep = np.load(keep_path).astype(int)  # 1-based
            t = t_all[np.clip(keep-1, 0, len(t_all)-1)]
            if len(t) == n_feats:
                return t
        except Exception:
            pass

    if len(t_all) == n_feats:
        return t_all

    t_end = float(t_all[-1]) if len(t_all) else float(n_feats-1)
    return np.linspace(0.0, t_end, num=n_feats, dtype=float)

# ------------------------------------ CLIP TEXT ENCODER --------------------------------
_CLIP_RT = {"model": None, "tokenizer": None, "device": None, "backend": None}
def ensure_clip_text_encoder(model_name='ViT-B-32', pretrained='openai'):
    if _CLIP_RT["model"] is not None: return
    try:
        import open_clip, torch
        device = "cuda" if torch.cuda.is_available() else "cpu"
        model, _, _ = open_clip.create_model_and_transforms(model_name, pretrained=pretrained)
        tokenizer = open_clip.get_tokenizer(model_name)
        model.to(device); model.eval()
        _CLIP_RT.update({"model": model, "tokenizer": tokenizer, "device": device, "backend": "open_clip"}); return
    except Exception:
        pass
    try:
        import clip, torch
        device = "cuda" if torch.cuda.is_available() else "cpu"
        model, _ = clip.load(model_name, device=device, jit=False)
        _CLIP_RT.update({"model": model, "tokenizer": clip.tokenize, "device": device, "backend": "clip"}); return
    except Exception as e:
        raise RuntimeError("Không tìm thấy text encoder CLIP (open_clip/clip).")

def encode_text(texts: List[str]) -> np.ndarray:
    ensure_clip_text_encoder()
    import torch
    toks = _CLIP_RT["tokenizer"](texts)
    with torch.no_grad():
        toks = toks.to(_CLIP_RT["device"])
        feats = _CLIP_RT["model"].encode_text(toks)
        feats = feats / feats.norm(dim=-1, keepdim=True)
    return feats.detach().cpu().numpy().astype("float32")

# ------------------------------------ LLM: GEMINI ASPECT/SEED --------------------------------
# Định nghĩa fallback thủ công nếu không dùng LLM
FALLBACK = {
    "aspects": [
        {"name": "grand sand sculpture", "prompts": [
            "grand sand sculpture at sand sculpture festival, monumental sand art, detailed figures",
            "large sand sculpture exhibit, festival display, intricate sand carving"
        ], "weight": 1.0},
        {"name": "rollerblader + two skateboarders", "prompts": [
            "youth street sports in sand sculpture: one rollerblader and two skateboarders",
            "skaters carved in sand: inline skates and two skateboards"
        ], "weight": 1.35},
        {"name": "arched backdrop with carved letters", "prompts": [
            "large arched backdrop with carved letters, inscriptions in sand",
            "curved arch structure with text engraved"
        ], "weight": 0.9},
        {"name": "multiple sand artworks", "prompts": [
            "multiple sand sculptures on display, exhibition with several sand artworks",
            "variety of sand art pieces shown together at festival"
        ], "weight": 0.8},
        {"name": "two pink smoke columns", "prompts": [
            "two pink smoke columns, pink smoke flares near sand sculptures",
            "pink colored smoke pillars beside sand sculpture"
        ], "weight": 1.35}
    ],
    "seeds": {
        "unigrams": ["điêu","khắc","cát","lễ","hội","tác","phẩm","trượt","patin","ván","skate","skateboard",
                     "vòm","cong","chữ","khói","hồng","cột","màu","thanh","niên"],
        "phrases":  ["điêu khắc cát","lễ hội điêu khắc","tác phẩm điêu khắc","trượt patin","trượt ván",
                     "vòm cong","khắc chữ","khói hồng","khói màu hồng","cột khói"]
    }
}

def extract_aspects_and_seeds_with_gemini(query_vi: str) -> Dict:
    """
    Trả về dict:
    {
      "aspects": [{"name": str, "prompts": [str,...], "weight": float}, ...],   # 1..N aspects linh hoạt
      "seeds": {"unigrams":[...], "phrases":[...]}
    }
    """
    if not (USE_LLM_ASPECTS and GEMINI_API_KEY):
        return FALLBACK

    sys_inst = (
            "Bạn là công cụ xây dựng truy vấn cho video retrieval.\n"
            "- Đầu vào: miêu tả TỰ NHIÊN (tiếng Việt/Anh).\n"
            "- Đầu ra: JSON với hai trường: 'aspects' và 'seeds'.\n"
            "  * aspects: mảng 1..N phần tử. Mỗi phần tử có:\n"
            "      - name: tên ngắn gọn (<= 6 từ)\n"
            "      - prompts: 2-4 câu tiếng Anh ngắn gọn để encode bằng CLIP\n"
            "      - weight: số thực >0 (mặc định 1.0). Tăng lên nếu cue rất đặc thù.\n"
            "  * seeds: { 'unigrams': [tối đa 28], 'phrases': [tối đa 18] } bằng TIẾNG VIỆT để dò transcript.\n"
            "Yêu cầu: JSON hợp lệ, không thêm bình luận khác."
        )

    payload = {"query_vi": query_vi}
    
    try:
        from google import genai
        client = genai.Client(api_key=GEMINI_API_KEY)
        
        try:
            resp = client.responses.generate(
                model=GEMINI_MODEL,
                input=json.dumps(payload, ensure_ascii=False),
                system_instruction=sys_inst,
                config={"max_output_tokens": GEMINI_MAX_TOKENS,
                        "response_mime_type": "application/json"}
            )
            txt = getattr(resp, "output_text", None)
            if not txt:
                raise RuntimeError("Empty output_text")
        except Exception:
            # ---- Đường gọi tương thích: API cũ (models.generate_content) ----
            cfg = genai.types.GenerateContentConfig(
                max_output_tokens=GEMINI_MAX_TOKENS,
                response_mime_type="application/json",
                system_instruction=sys_inst
            )
            resp = client.models.generate_content(
                model=GEMINI_MODEL,
                contents=json.dumps(payload, ensure_ascii=False),
                config=cfg
            )
            txt = getattr(resp, "text", None) or getattr(resp, "output_text", None)
            if not txt:
                raise RuntimeError("Empty text")

        data = json.loads(txt)
        if "aspects" not in data or not isinstance(data["aspects"], list):
            raise ValueError("no aspects")
        if "seeds" not in data or not isinstance(data["seeds"], dict):
            data["seeds"] = {"unigrams": [], "phrases": []}
        return data
    except Exception as e:
        print("[Gemini] Fallback:", e)
        return FALLBACK

# ------------------------------------ BUILD QUERY VECTORS --------------------------------
def build_query_vecs_for_aspects(aspects: List[Dict]) -> Tuple[List[np.ndarray], List[str], List[float]]:
    all_prompts, ranges, names, weights = [], [], [], []
    for a in aspects:
        ps = list(dict.fromkeys(a.get("prompts", [])))  # unique & keep order
        s = len(all_prompts)
        all_prompts.extend(ps)
        ranges.append((s, len(all_prompts)))
        names.append(a.get("name","aspect"))
        weights.append(float(a.get("weight", 1.0)))
    if not all_prompts:
        # ensure at least one vec to avoid crash
        all_prompts = ["generic scene"]
        ranges = [(0,1)]
        names = ["generic"]
        weights = [1.0]

    q = encode_text(all_prompts).astype("float32")
    q = l2_normalize_rows(q)
    q_per_aspect = [q[s:e] for (s, e) in ranges]
    return q_per_aspect, names, weights

# ------------------------------------ WINDOW SCORING -------------------------------------
def best_window_multi_aspect(feats: np.ndarray, times: np.ndarray,
                             q_per_aspect: List[np.ndarray],
                             aspect_weights: List[float],
                             window_sec: float = 40.0,
                             min_start_sec: float = 0.0):
    N = feats.shape[0]
    if N == 0:
        return {"score": -1e9, "i0": 0, "i1": 0, "t0": 0.0, "t1": 0.0,
                "aspect_frames": [1]*len(q_per_aspect), "aspect_scores": [0.0]*len(q_per_aspect)}

    start_idx = int(np.searchsorted(times, min_start_sec, side="left"))
    start_idx = min(max(start_idx, 0), N-1)
    feats = feats[start_idx:]; times = times[start_idx:]; Nsub = feats.shape[0]

    # sims per aspect (max over prompt variants)
    sims_list = []
    for qA in q_per_aspect:
        sims = (qA @ feats.T) if len(qA) > 0 else np.zeros((1, Nsub), dtype="float32")
        sims_list.append(sims.max(axis=0).astype("float32"))

    deques = [deque() for _ in range(len(sims_list))]
    best = {"score": -1e9, "i0": 0, "i1": 0, "peaks": [(0,0.0)]*len(sims_list)}
    e = 0
    for i in range(Nsub):
        while e < Nsub and (times[e] - times[i] <= window_sec):
            for j, s in enumerate(sims_list):
                dq = deques[j]; val = float(s[e])
                while dq and dq[-1][1] <= val: dq.pop()
                dq.append((e, val))
            e += 1
        win_sum, peaks = 0.0, []
        for j, dq in enumerate(deques):
            idx, val = (dq[0] if dq else (i, float("-inf")))
            peaks.append((idx, val)); win_sum += float(aspect_weights[j])*val
        if win_sum > best["score"]: best = {"score": win_sum, "i0": i, "i1": e-1, "peaks": peaks}
        for dq in deques:
            if dq and dq[0][0] == i: dq.popleft()

    aspect_frames = [int(start_idx + idx + 1) for (idx, _v) in best["peaks"]]
    aspect_scores = [float(_v) for (_idx, _v) in best["peaks"]]
    i0, i1 = best["i0"], best["i1"]
    t0 = float(times[i0]); t1 = float(times[min(i1, len(times)-1)])
    return {"score": float(best["score"]),
            "i0": int(start_idx + i0), "i1": int(start_idx + i1),
            "t0": t0, "t1": t1,
            "aspect_frames": aspect_frames, "aspect_scores": aspect_scores}

def score_video_clip_window(video_id: str,
                            q_per_aspect: List[np.ndarray],
                            aspect_weights: List[float],
                            window_sec=40.0, min_start_sec=0.0):
    npy_path = npy_path_for_video(video_id)
    if not os.path.exists(npy_path):
        return {"video_id": video_id, "ok": False, "error": "no npy"}
    feats = np.load(npy_path).astype("float32")
    feats = l2_normalize_rows(feats)
    times = load_times_for_video(video_id, len(feats))
    win = best_window_multi_aspect(feats, times, q_per_aspect, aspect_weights,
                                   window_sec=window_sec, min_start_sec=min_start_sec)
    return {"video_id": video_id, "ok": True, "total": win["score"], "win": win}

def rank_videos_clip_window(aspects: List[Dict],
                            limit_videos=None, window_sec=40.0, min_start_sec=0.0,
                            include_series: set | None = None,
                            exclude_series: set | None = None,
                            include_regex: str | None = None,
                            exclude_regex: str | None = None,
                            include_ids: set | None = None,
                            exclude_ids: set | None = None):
    clip_dirs = []
    if "b1" in DATASETS: clip_dirs.append(B1["clip_dir"])
    if "b2" in DATASETS: clip_dirs.append(B2["clip_dir"])

    vids = list_videos_from_dirs(
        clip_dirs,
        limit=limit_videos,
        include_series=include_series, exclude_series=exclude_series,
        include_regex=include_regex, exclude_regex=exclude_regex,
        include_ids=include_ids, exclude_ids=exclude_ids
    )

    q_per_aspect, names, weights = build_query_vecs_for_aspects(aspects)
    out = []
    for vid in vids:
        r = score_video_clip_window(vid, q_per_aspect, weights, window_sec=window_sec, min_start_sec=min_start_sec)
        if r["ok"]: out.append(r)
    out.sort(key=lambda x: x["total"], reverse=True)
    return out, names, weights

# ------------------------------------ VISUALIZATION -------------------------------------
def show_grid_with_videos(results, aspect_names: List[str],
                          top_videos=10, figsize=None):
    rows = min(top_videos, len(results))
    cols = len(aspect_names)
    if figsize is None:
        # rộng hơn & dãn cách để tránh chồng chữ
        figsize = (max(20, cols*5), max(8, rows*3.6))

    fig, axes = plt.subplots(rows, cols, figsize=figsize, squeeze=False)
    for i in range(rows):
        r = results[i]; vid, win = r["video_id"], r["win"]
        frames, scores = win["aspect_frames"], win["aspect_scores"]
        for j in range(cols):
            ax = axes[i, j]
            img = safe_imread(keyframe_path(vid, int(frames[j])))
            ax.imshow(img)
            if j == 0:
                t0s, t1s = sec_to_mmss(win['t0']), sec_to_mmss(win['t1'])
                title = f"{vid}  |  total={r['total']:.3f}\nwin: {t0s} → {t1s}"
                ax.set_title(title, fontsize=10, pad=8)
            # nhãn dưới có thêm khoảng trống
            label = f"{aspect_names[j]}\nframe={frames[j]}   s={scores[j]:.3f}"
            ax.set_xlabel(label, fontsize=9, labelpad=8)
            ax.set_xticks([]); ax.set_yticks([])
    # dãn cách
    plt.tight_layout(pad=1.3, rect=(0,0,1,1))
    plt.subplots_adjust(hspace=0.42, wspace=0.15)
    plt.show()

# ------------------------------------ TEXT RERANK (ULTRA-WINDOWED) ----------------------
def attach_text_dbs(conn, attach_b1=True, attach_b2=True):
    cur = conn.cursor()
    existing = {row[1] for row in cur.execute("PRAGMA database_list").fetchall()}
    if attach_b1 and Path(B1_DB_PATH).exists() and "b1" not in existing:
        cur.execute("ATTACH DATABASE ? AS b1;", (B1_DB_PATH,))
    if attach_b2 and Path(B2_DB_PATH).exists() and "b2" not in existing:
        cur.execute("ATTACH DATABASE ? AS b2;", (B2_DB_PATH,))
    conn.commit()

def _tok_vi(s): return re.findall(r"[0-9]+|[A-Za-zÀ-ỹĐđ]+", s.lower())

def build_keyword_sets(query_vi: str, seeds: Dict, max_unigrams=28, max_phrases=18):
    toks = _tok_vi(query_vi)
    unigrams = {t for t in toks if len(t) >= 2}
    phrases  = {" ".join(toks[i:i+2]) for i in range(len(toks)-1)}
    # bổ sung từ seeds
    unigrams.update(seeds.get("unigrams", []))
    phrases.update(seeds.get("phrases", []))
    uni = sorted(list(unigrams))[:max_unigrams]
    phr = sorted(list(phrases))[:max_phrases]
    q_any = " OR ".join(uni + [f'"{p}"' for p in phr]) if (uni or phr) else None
    q_phr = " OR ".join([f'"{p}"' for p in phr]) if phr else None
    return uni, phr, q_any, q_phr

def _ph(n): return ",".join(["?"]*n)

def create_temp_window_table(conn, cand_rows, delta_sec=20.0):
    conn.execute("DROP TABLE IF EXISTS temp.win;")
    conn.execute("CREATE TABLE temp.win (video_id TEXT PRIMARY KEY, t0 REAL, t1 REAL);")
    for r in cand_rows:
        t0 = max(0.0, float(r["t0"]) - float(delta_sec))
        t1 = float(r["t1"]) + float(delta_sec)
        conn.execute("INSERT INTO temp.win(video_id, t0, t1) VALUES (?, ?, ?);", (r["video_id"], t0, t1))
    conn.commit()

def text_hits_window_for_alias(conn, alias: str, q_any: str | None, q_phr: str | None,
                               only_vids: list[str]):
    if not only_vids:
        return pd.DataFrame(columns=["video_id","hits_any","hits_phr"])

    cur = conn.cursor()
    ph  = ",".join(["?"] * len(only_vids))

    # any
    rows_any = []
    if q_any:
        sql_any = f"""
        SELECT d.video_id, COUNT(*) AS hits
        FROM {alias}.docs AS d
        JOIN temp.win w ON d.video_id = w.video_id
        WHERE d.video_id IN ({ph})
          AND d.t BETWEEN w.t0 AND w.t1
          AND docs MATCH ?
        GROUP BY d.video_id;"""
        rows_any = cur.execute(sql_any, (*only_vids, q_any)).fetchall()

    # phrases
    rows_phr = []
    if q_phr:
        sql_phr = f"""
        SELECT d.video_id, COUNT(*) AS hits
        FROM {alias}.docs AS d
        JOIN temp.win w ON d.video_id = w.video_id
        WHERE d.video_id IN ({ph})
          AND d.t BETWEEN w.t0 AND w.t1
          AND docs MATCH ?
        GROUP BY d.video_id;"""
        rows_phr = cur.execute(sql_phr, (*only_vids, q_phr)).fetchall()

    any_map = {vid: int(h) for vid, h in rows_any}
    phr_map = {vid: int(h) for vid, h in rows_phr}
    out = [(vid, any_map.get(vid,0), phr_map.get(vid,0)) for vid in only_vids]
    return pd.DataFrame(out, columns=["video_id","hits_any","hits_phr"])

def text_hits_window_for_alias_per_video(conn, alias: str, windows, q_any: str|None, q_phr: str|None):
    """
    windows: list[{'video_id', 't0', 't1'}]
    Trả về DataFrame(video_id, hits_any, hits_phr).
    """
    cur = conn.cursor()
    rows = []
    sql_base = f"SELECT COUNT(*) FROM {alias}.docs WHERE video_id=? AND t BETWEEN ? AND ? AND docs MATCH ?"
    for w in windows:
        vid, t0, t1 = w["video_id"], float(w["t0"]), float(w["t1"])
        h_any = h_phr = 0
        if q_any:
            h_any = cur.execute(sql_base, (vid, t0, t1, q_any)).fetchone()[0]
        if q_phr:
            h_phr = cur.execute(sql_base, (vid, t0, t1, q_phr)).fetchone()[0]
        rows.append((vid, int(h_any), int(h_phr)))
    return pd.DataFrame(rows, columns=["video_id","hits_any","hits_phr"])

# Kết nối read-only + immutable (SQLite tối ưu đọc)
def _connect_ro(db_path: str):
    uri = f"file:{db_path}?immutable=1&mode=ro"
    return sqlite3.connect(uri, uri=True, check_same_thread=False)

def _sqlite_speed_pragmas(conn):
    cur = conn.cursor()
    try:
        cur.execute("PRAGMA cache_size=-400000;")   # ~400MB nếu đủ RAM
        cur.execute("PRAGMA temp_store=MEMORY;")
        cur.execute("PRAGMA mmap_size=268435456;")  # 256MB
        cur.execute("PRAGMA optimize;")
        conn.commit()
    except Exception:
        pass

def _count_one(db_path: str, vid: str, t0: float, t1: float, q_any: str|None, q_phr: str|None):
    conn = _connect_ro(db_path); _sqlite_speed_pragmas(conn)
    cur  = conn.cursor()
    sql  = "SELECT COUNT(*) FROM docs WHERE video_id=? AND t BETWEEN ? AND ? AND docs MATCH ?"
    h_any = h_phr = 0
    if q_any:
        h_any = cur.execute(sql, (vid, t0, t1, q_any)).fetchone()[0]
    if q_phr:
        h_phr = cur.execute(sql, (vid, t0, t1, q_phr)).fetchone()[0]
    conn.close()
    return (vid, int(h_any), int(h_phr))

def text_hits_window_for_alias_per_video_mt(db_path: str, windows, q_any: str|None, q_phr: str|None, workers: int = 8):
    """
    windows: list[{'video_id','t0','t1'}] — đếm theo từng video, chạy đa luồng.
    """
    rows = []
    # Giới hạn workers hợp lý (Kaggle thường 2 CPU; 8 vẫn ổn vì chủ yếu I/O)
    workers = max(1, workers)
    with concurrent.futures.ThreadPoolExecutor(max_workers=workers) as ex:
        futures = [ex.submit(_count_one, db_path, w["video_id"], float(w["t0"]), float(w["t1"]), q_any, q_phr)
                   for w in windows]
        for fut in concurrent.futures.as_completed(futures):
            rows.append(fut.result())
    return pd.DataFrame(rows, columns=["video_id","hits_any","hits_phr"])

# --- gom window theo alias và map ra path DB ---
def _alias_windows_and_dbpaths(windows):
    g = {"b1": [], "b2": []}
    for w in windows:
        (g["b1"] if w["video_id"].startswith("L") else g["b2"]).append(w)
    out = []
    if g["b1"] and os.path.exists(B1_DB_PATH): out.append(("b1", B1_DB_PATH, g["b1"]))
    if g["b2"] and os.path.exists(B2_DB_PATH): out.append(("b2", B2_DB_PATH, g["b2"]))
    return out

# -- Helper: gom windows theo alias DB (b1 cho L*, b2 cho K*)
def group_windows_by_alias(conn, windows):
    out = {"b1": [], "b2": []}
    for w in windows:
        (out["b1"] if w["video_id"].startswith("L") else out["b2"]).append(w)
    existing = {row[1] for row in conn.cursor().execute("PRAGMA database_list").fetchall()}
    return {al: lst for al, lst in out.items() if al in existing and lst}

# -- Helper: PRAGMA tăng tốc (tùy RAM)
def _apply_sqlite_pragmas(conn):
    cur = conn.cursor()
    try:
        cur.execute("PRAGMA cache_size=-400000;")   # ~400MB
        cur.execute("PRAGMA temp_store=MEMORY;")
        cur.execute("PRAGMA mmap_size=268435456;")  # 256MB
        cur.execute("PRAGMA optimize;")
        conn.commit()
    except Exception:
        pass

# ---- SỬ DỤNG: thay thế phần trong rerank_with_transcript_ultra_window() ----
def rerank_with_transcript_ultra_window(
    clip_results: list,
    query_vi: str,
    seeds: dict,
    top_candidates: int = 40,
    delta_sec: float = 10.0,
    w_clip: float = 0.6, w_text: float = 0.4,
    text_weights: dict | None = None,
    verbose: bool = True, timing: bool = True,
    *,
    use_per_video_match: bool = True,     # vẫn giữ tham số để tương thích
    workers: int = 8,                     # <--- số luồng
    phrases_only: bool = False,           # <--- chỉ dùng phrase để nhanh hơn
    max_unigrams: int = 16, max_phrases: int = 10
):
    if text_weights is None:
        text_weights = {"any": 0.5, "phr": 0.5}

    t0 = time.perf_counter()
    cand = [{"video_id": r["video_id"], "clip_total": float(r["total"]),
             "t0": max(0.0, float(r["win"]["t0"]) - float(delta_sec)),
             "t1": float(r["win"]["t1"]) + float(delta_sec)}
            for r in clip_results[:top_candidates]]
    df = pd.DataFrame(cand)
    if df.empty: return df

    # Rút gọn term
    uni, phr, q_any, q_phr = build_keyword_sets(
        query_vi, seeds,
        max_unigrams=(0 if phrases_only else max_unigrams),
        max_phrases=max_phrases
    )
    if phrases_only:  # bỏ q_any để MATCH nhẹ nhất
        q_any = None

    # Đếm đa luồng theo từng video & từng alias DB
    parts = []
    for al, db_path, wins in _alias_windows_and_dbpaths(cand):
        parts.append(text_hits_window_for_alias_per_video_mt(db_path, wins, q_any, q_phr, workers=workers))
    text_df = (pd.concat(parts, ignore_index=True)
               if parts else pd.DataFrame(columns=["video_id","hits_any","hits_phr"]))

    # Fuse điểm
    df = df.merge(text_df, on="video_id", how="left").fillna({"hits_any":0,"hits_phr":0})
    def _minmax(x):
        x = x.astype(float); mn, mx = x.min(), x.max()
        if mx - mn < 1e-9: return np.ones_like(x, dtype=float)
        return (x - mn) / (mx - mn)
    df["clip_n"] = _minmax(df["clip_total"].values)
    df["any_n"]  = _minmax(df["hits_any"].values)
    df["phr_n"]  = _minmax(df["hits_phr"].values)
    a, b = text_weights["any"], text_weights["phr"]
    df["text_score"] = a*df["any_n"] + b*df["phr_n"]
    df["fused"]      = w_clip*df["clip_n"] + w_text*df["text_score"]
    df = df.sort_values("fused", ascending=False).reset_index(drop=True)

    if verbose:
        print(f"[ULTRA-WIN] fused = {w_clip}*clip + {w_text}*( {a}*hits_any + {b}*hits_phr ) | Δ={delta_sec}s | workers={workers} | phrases_only={phrases_only}")
        head = df.head(10).copy()
        head["t0"] = [sec_to_mmss(t) for t in head["t0"].values]
        head["t1"] = [sec_to_mmss(t) for t in head["t1"].values]
        display(head[["video_id","clip_total","hits_any","hits_phr","clip_n","text_score","fused","t0","t1"]])

    if timing:
        print(f"[ULTRA-WIN] elapsed: {time.perf_counter()-t0:.3f}s (candidates={len(df)})")
    return df

def reorder_results_by_df(orig_results, ranked_df, top_videos=10):
    order = ranked_df.head(top_videos)["video_id"].tolist()
    vid2r = {r["video_id"]: r for r in orig_results}
    return [vid2r[v] for v in order if v in vid2r]

In [ ]:
spec = extract_aspects_and_seeds_with_gemini(QUERY_VI)
ASPECTS = spec["aspects"]
SEEDS   = spec.get("seeds", {})
print("Using", len(ASPECTS), "aspects:")
for a in ASPECTS:
    print(" -", a["name"], f"(w={a.get('weight',1.0)})")
print(spec)

# Start search here

In [ ]:
# ======================================== RUN ===========================================
# Query ví dụ (có thể thay đổi liên tục, LLM sẽ linh hoạt số lượng aspects)
QUERY_VI = """
Trong đoạn clip có là một chàng trai đội mũ lưỡi trai đen, mặc áo thun trắng có dòng chữ tiếng Anh, đang ngồi cạnh một chiếc hộp. 
Trên hộp, anh sắp xếp nhiều mảnh bìa cắt rời với hình thù ngẫu nhiên.
Nhờ ánh sáng chiếu từ một phía, các mảnh bìa đó đổ bóng lên tường, tạo thành hình chân dung một người đàn ông với khuôn mặt rõ nét, tóc vuốt gọn và mặc vest.
"""

# 0) Sinh aspects & seeds
spec = extract_aspects_and_seeds_with_gemini(QUERY_VI)
ASPECTS  = spec["aspects"]          # [{name, prompts[], weight}, ...]
SEEDS    = spec.get("seeds", {})

print(f"Using {len(ASPECTS)} aspects:")
for a in ASPECTS:
    print(" -", a["name"], f"(w={a.get('weight',1.0)})")

# 1) CLIP rank
results, aspect_names, aspect_weights = rank_videos_clip_window(
    ASPECTS,
    limit_videos=LIMIT_VIDEOS,
    window_sec=WINDOW_SEC,
    min_start_sec=SKIP_SEC,
    exclude_series=EXCLUDE_SER
)

print(f"Scored videos: {len(results)}")
for i, r in enumerate(results[:TOP_VIDEOS]):
    t0s, t1s = sec_to_mmss(r['win']['t0']), sec_to_mmss(r['win']['t1'])
    print(f"[{i}]", r["video_id"], "| total:", round(r["total"],3),
          "| win:", f"{t0s}→{t1s}",
          "| frames:", r["win"]["aspect_frames"])

show_grid_with_videos(results, aspect_names=aspect_names, top_videos=TOP_VIDEOS, figsize=None)

# 2) Transcript rerank (windowed)
reranked = rerank_with_transcript_ultra_window(
    results, QUERY_VI, SEEDS,
    top_candidates=40, delta_sec=DELTA_SEC,
    w_clip=0.6, w_text=0.4,
    text_weights={"any":0.5,"phr":0.5},
    use_per_video_match=True,            # <— bật nhánh nhanh
    max_unigrams=16, max_phrases=10
)

# 3) Vẽ lại grid theo thứ hạng fused
top_after = reorder_results_by_df(results, reranked, top_videos=TOP_VIDEOS)
show_grid_with_videos(top_after, aspect_names=aspect_names, top_videos=TOP_VIDEOS, figsize=None)